# Structural Estimation using DC-EGM for Education Level 1

## Dependencies, Packages & Roots

### Magics

In [ ]:
%reload_ext autoreload
%autoreload 2

### Project Paths

In [ ]:
from pathlib import Path
import sys
import importlib

# Change this depending on notebook location:
# 0 = notebook is 1 folder inside project root, e.g. dp_termpaper/edu1
# 1 = notebook is 2 folders inside project root, e.g. dp_termpaper/data/moments
# 2 = notebook is 3 folders inside project root
amount_of_levels = 0

DIR = Path.cwd().resolve().parents[amount_of_levels]

if str(DIR) not in sys.path:
    sys.path.insert(0, str(DIR))

import project_paths as pp
from project_imports import *

importlib.reload(pp)
jax.config.update("jax_enable_x64", True)

### Determine Macros

In [ ]:
###############################################################
#### Define which education level you want to estimate for ####
###############################################################

education_level = "edu1"
# education_level = "edu2"
# education_level = "edu3"

# Amount of individuals in simulated sample
n_sim = 1
n_individuals = 100_000

# Whether to run the structural estimation or just load the results
structural_estimation = True
# structural_estimation = False

### Load Data


In [ ]:

education_level_file = pp.MOMENTS_DIR / f"moments_{education_level}.txt"

MORTALITY = pp.DATA_DIR / "mortality.xlsx"

beta0, beta1, beta2 = np.loadtxt(pp.FIRST_STAGE_RESULTS_DIR / f"wage_params_{education_level}_ols.txt")
alpha1, alpha2 = np.loadtxt(pp.STRUCTURAL_RESULTS_DIR/ "mortality_params.txt")

In [ ]:
# Read CSV
df_edu = pd.read_csv(education_level_file)
# read mortality and discard seoncd and third column
df_mort = pd.read_excel(MORTALITY, sheet_name="DOD", usecols=[0, 3])

# 2) standardize colomn name and remove _FREQ_ column
# Iterate over DataFrames (if there are multiple DataFrames to process)
for data_frame in [df_edu]:
    # Rename ALDER → age
    if "ALDER" in data_frame.columns:
        data_frame.rename(columns={"ALDER": "age"}, inplace=True)
    # Remove _FREQ_-column if it exists
    # if "_FREQ_" in data_frame.columns:
    #     data_frame.drop(columns=["_FREQ_"], inplace=True)
# drop var_wage, skew_wage and pens
df_edu.drop(columns=["var_wage", "skew_wage", "pens"], errors="ignore", inplace=True)
# df_edu

## Options for model - Choices and states

In [ ]:
n_periods = 55
labour_choices = np.arange(5) # 5 choices

model_config = {
    "n_periods": n_periods,
    "choices": labour_choices, # 5 choices
    "n_quad_points": 5,
    "continuous_states": {
        "assets_end_of_period": np.linspace(0, 50, 20),
        "experience": jnp.linspace(0, 1, 5).astype(float) # 1 experince grid point, if experience can only go up by a year - more points if it is a fraction based on hours worked
        },
    "stochastic_states": {
        "survival": [0, 1],
    }
    }

model_specs = {
    "choices": labour_choices, # 5 choices
    "n_periods": n_periods,
    "labour_choices": labour_choices, # 5 choices
    "hours": jnp.array([0,250,750,1300,1900]), #list 
    "max_hours": 1900,
    "start_age": 30,
    "tax_threshold1": 0.480,
    "tax_threshold2": 5.698,
    "tax_base_rate": 0.38,
    "tax_top_rate": 0.5,
    "retirement_age": 67,
    "oap_base_amount": 0.80328,
    "oap_max_supplement": 0.92940,
    "supp_threshold": 0.79300,
    "oap_threshold": 3.3592,
    "supp_reduction_rate": 0.309,
    "oap_reduction_rate": 0.3,
    "alpha1": alpha1, # independently estimated parameter for survival probability
    "alpha2": alpha2,  # independently estimated parameter for survival probability
    "max_init_experience": 5,
    "max_ret_period": 45, # Age 75
    "min_ret_period": 30, # Age 60
}
stochastic_states_transitions = {
    "survival": prob_survival
}

### Setup the model

In [ ]:
model = dcegm.setup_model(
    model_config=model_config,
    model_specs=model_specs,
    utility_functions=new_utility_functions,
    utility_functions_final_period=final_period_utility,
    budget_constraint=budget_dcegm_initial,
    state_space_functions=create_state_space_function_dict(),
    stochastic_states_transitions=stochastic_states_transitions,
)

# Structural Estimation

### Initial values for simulating

In [ ]:
seed = 132
key = jax.random.PRNGKey(0)
n = n_individuals       

# hours mapping for simulation
hours_map = {0: 0, 1: 250, 2: 750, 3: 1300, 4: 1900}

#age for simulation
start_age = model_specs["start_age"] 

# distribution of initial choices
labels = jnp.array([0, 1, 2, 3, 4], dtype=jnp.int32)
probs  = jnp.array([0.259155, 0.118310, 0.108099, 0.108451, 0.405986])  # sums to 1.0

lagged_choice = jax.random.choice(
    key,
    a       = labels,
    shape   = (n,),
    p       = probs,
    replace = True
)

# set initial states for each individual
states_initial = {
    "period": jnp.zeros(n_individuals, dtype=jnp.int32),      # Every individual starts at period 0 (age 30)
    "lagged_choice": lagged_choice,  # Every individual starts with choice 3 (work fulltime)
    "experience": jnp.full(n_individuals, 1).astype(float),  # Every individual starts with 5 years of experience
    "survival": jnp.ones(n_individuals, dtype=jnp.int32), # Every individual starts with 1 (alive)
    "assets_begin_of_period": jnp.full(n_individuals, 0.505047) # Every individual starts with 59k wealth - to be adjusted based on the actual moments
}

### Initiatal parameters

In [ ]:
params_initial = load_params_txt(pp.STRUCTURAL_RESULTS_DIR / f"optimized_params_phi_{education_level}.txt")
params_initial["gamma"] = jnp.asarray(params_initial["gamma"])

# Estimation

In [ ]:
# keep_cols = ["hours_0", "hours_1","hours_2","hours_3","hours_4", "work_work", "nowork_nowork"]
all_cols =  ["hours_0", "hours_1", "hours_2", "hours_3", "hours_4", "work_work", "nowork_nowork", "avg_wealth", "avg_experience", "avg_labor_income", "avg_wage", "avg_hours"]
keep_cols = all_cols  # Keep all moments for now, but you can adjust this list to focus on specific moments
share_cols = keep_cols

In [ ]:
# Helper function for params into critical function
def theta_to_params(theta_array, base_params):
    p = base_params.copy()

    p["interest_rate"] = 0.01
    p["beta0"] = beta0
    p["beta1"] = beta1
    p["beta2"] = beta2

    p["income_shock_mean"] = theta_array[0]
    p["income_shock_std"] = theta_array[1]
    p["taste_shock_scale"] = theta_array[2]
    p["discount_factor"] = theta_array[3]
    p["rho"] = theta_array[4]
    p["gamma"] = jnp.array([
        theta_array[5],
        theta_array[6],
        theta_array[7],
        theta_array[8],
    ])
    p["kappa1"] = theta_array[9]
    p["kappa2"] = theta_array[10]

    p["phi_entry"] = theta_array[11]
    p["phi_exit"] = theta_array[12]
    p["phi_switch"] = theta_array[13]

    p["b_scale"] = theta_array[14]
    p["xi"] = theta_array[15]

    p.pop("phi", None)
    
    return p

# Helper function for determining variance in empirical moments
def compute_empirical_share_variances(df, share_cols, freq_col="_FREQ_"):
    df_var = df[["age", freq_col] + share_cols].copy()

    for col in share_cols:
        p = df_var[col]
        n = df_var[freq_col]
        df_var[col] = p * (1 - p) / n

    return df_var

In [ ]:
base_params = params_initial.copy()
empirical_var = compute_empirical_share_variances(df_edu, share_cols)

In [ ]:
# Function for Estimation in structural estimation file
def crit_func_scipy(theta_array):
    """
    High-level objective function for MSM that updates all parameters,
    runs the simulation, computes simulated moments and compares them
    to empirical moments. The objective is the weighted sum of squared differences,
    where the weights are given by the inverse of the variances of the empirical moments.
    
    theta_array: a numpy array with 16 elements corresponding to:
       [income_shock_mean, income_shock_std, taste_shock_scale, beta, rho, gamma1, gamma2, gamma3,
        beta0, beta1, beta2, kappa1, kappa2, phi_entry, phi_exit, phi_switch]
    """
    
    # Convert theta vector into clean parameter dictionary
    p = theta_to_params(theta_array, base_params=base_params)

    print("Parameter dictionary used:")
    for key, value in p.items():
        print(f"{key}: {value}")

    # Validate exogenous parameters
    model.validate_exogenous(p)

    # Solve model at current parameter vector
    model_solved = model.solve(p)

    # Simulate model
    sim = model_solved.simulate(
        states_initial=states_initial,
        seed=seed,
    )

    # df_sim = create_simulation_df(sim)

    # Compute simulated moments - returning a dataframe similar to the actual data
    sim_moments = compute_simulation_moments(sim, start_age, hours_map)
    empirical_moms = df_edu  # Your empirical moments DataFrame

    # Drop the "pens" column from both simulated and empirical moments if present
    if "pens" in sim_moments.columns:
        sim_moments = sim_moments.drop(columns=["pens"])
    if "pens" in empirical_moms.columns:
        empirical_moms = empirical_moms.drop(columns=["pens"])

    # Select only the columns of interest for the moments comparison
    sim_moments = sim_moments[keep_cols]
    empirical_moms = empirical_moms[keep_cols]
    # Convert DataFrames to numpy arrays.
    # They should have the same shape, e.g., (num_ages, num_moments)
    sim_vals = sim_moments.to_numpy()
    emp_vals = empirical_moms.to_numpy()

    # Debug-print shapes so that we can ensure they are the same shape
    print("Simulated moments shape:", sim_vals.shape)
    print("Empirical moments shape:", emp_vals.shape)

        # Compute the difference between simulated and empirical moments.
    # diff will have shape (num_ages, num_moments)
    diff = sim_vals - emp_vals

    # Now, for each moment (each column), compute the sample variance of the empirical moment
    ## Should be computed through the data before moments i think
    emp_var = np.nanvar(emp_vals, axis=0, ddof=1)  # Sample variance
    # compute simulated moments variance
    sim_var = np.nanvar(sim_vals, axis=0, ddof=1)  # Sample variance

    total_var = emp_var + sim_var

    # Set a small epsilon to avoid division by zero.
    epsilon = 1e-6

    # The weight for each moment is defined as 1/(variance + epsilon).
    weights = 1.0 / (emp_var + epsilon)
  
    # Instead of forming a full weighting matrix, we can compute the objective by summing over moment columns.
    # For each moment i, compute the sum of squared differences over ages, multiplied by its weight.
    # as in crit = diff^T * W * diff
    # where W is a diagonal matrix with weights on the diagonal.
    # The crit_val is the sum of the weighted squared differences.
    
    num_moments = diff.shape[1]
    crit_val = 0.0
    for i in range(num_moments):
        crit_val += weights[i] * np.nansum(diff[:, i] ** 2)
    
    print("Crit value:", crit_val)
    print('')
    return float(crit_val)

In [ ]:
if structural_estimation is True:
    flat_params = [
        params_initial["income_shock_mean"],
        params_initial["income_shock_std"],
        params_initial["taste_shock_scale"],
        params_initial["discount_factor"],
        params_initial["rho"],
        *params_initial["gamma"].tolist(),
        params_initial["kappa1"],
        params_initial["kappa2"],
        params_initial["phi_entry"],
        params_initial["phi_exit"],
        params_initial["phi_switch"],
        params_initial["b_scale"],
        params_initial["xi"],
    ]

    initial_guess = np.array(flat_params, dtype=float)
    result = minimize(crit_func_scipy, initial_guess, method="Nelder-Mead", options={'disp': True})
    print("Optimization result:")
    print(result)   

### Save the estimation results

In [ ]:
if structural_estimation is True:
    # Rebuild optimized parameter dictionary using the same mapping
    # as the criterion function.
    params_hat = theta_to_params(result.x, base_params=base_params)

    # Make absolutely sure legacy names are gone
    params_hat.pop("sigma", None)
    params_hat.pop("beta", None)
    params_hat.pop("phi", None)

    # Convert gamma from JAX array to NumPy array for cleaner saving    
    params_hat["gamma"] = np.asarray(params_hat["gamma"])

    # Save params dict as a .txt file
    params_df = pd.DataFrame.from_dict(
        params_hat,
        orient="index",
        columns=["value"],
    )

    params_df.to_csv(
        pp.STRUCTURAL_RESULTS_DIR / f"optimized_params_{education_level}.txt",
        sep="\t",
    )

    params_df

## Plots

### Solve, simulate and plot final model

In [ ]:
if structural_estimation is True:
    model_solved = model.solve(params_hat)
    sim = model_solved.simulate(
        states_initial=states_initial,
        seed=seed,
    )
elif structural_estimation is False:
    model_solved = model.solve(params_initial)
    sim = model_solved.simulate(
        states_initial=states_initial,
        seed=seed,
    )

In [ ]:
edu = df_edu.copy().loc[df_edu.age <= 75]       # empirical
# moments_sim = compute_simulation_moments_with_ci(sim, start_age, hours_map)  # Call the function
moments_sim = compute_simulation_moments_with_ci(sim, start_age, hours_map)  # Call the function

# Ensure moments_sim.age is a pandas Series before filtering
moments_sim = moments_sim.loc[pd.Series(moments_sim.age) <= 75]  # Filter simulated + CIs
# scale avg_wealth by 100_000

# save single plots
# plot_empirical_vs_simulated_with_ci(
#     edu = edu,
#     moments_sim = moments_sim,
#     out_subfolder=f"{education_level}_phi_specification")

plot_empirical_vs_simulated_with_ci_grid(
    edu=edu,
    moments_sim=moments_sim,
    out_subfolder=f"{education_level}_phi_specification",
    ncols=3,
    figsize_per_plot=(4, 4),
    filename=f"empirical_vs_simulated_grid_{education_level}.png",
)